# BASMEDSecure Disclosed Baseline

This notebook presents a bounded, repaired reproduction for research comparison. It is not the original published software. It is delivered without execution outputs. Use a fresh kernel and start Jupyter from the repository root, the `presentation` directory, or this notebook directory.

## 1. Scope

The baseline accepts unsigned 8-bit arrays, fits a fresh multinomial logistic-regression model to deterministic threshold labels, embeds through adaptive LSB replacement, and returns an external visited-prefix depth map. It does not support general DICOM files and does not recover the original cover.

In [ ]:
%pip install "numpy>=1.26,<3" "scikit-learn==1.7.2" "scikit-image>=0.24,<0.27" matplotlib

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

if any(name == "ichan_secure" or name.startswith("ichan_secure.") for name in sys.modules):
    raise RuntimeError("Use a fresh kernel before running this presentation notebook")

source_candidates = [
    Path.cwd() / "assets/source",
    Path.cwd() / "basmedsecure/assets/source",
    Path.cwd() / "presentation/basmedsecure/assets/source",
]
source_matches = [path.resolve() for path in source_candidates if (path / "ichan_secure/__init__.py").is_file()]
if len(source_matches) != 1:
    raise RuntimeError(f"Expected exactly one BASMEDSecure source tree, found: {source_matches}")
SOURCE_ROOT = source_matches[0]
sys.path.insert(0, str(SOURCE_ROOT))

import ichan_secure
module_path = Path(ichan_secure.__file__).resolve()
if SOURCE_ROOT not in module_path.parents:
    raise RuntimeError(f"Unexpected ichan_secure import: {module_path}")

from ichan_secure import Domain
from ichan_secure.baseline import embed_baseline, extract_baseline
from ichan_secure.metrics import direct_map_cost, distortion_metrics, recovery_metrics

## 2. Controlled input

The synthetic input below is a software fixture, not a clinical dataset. A later comparison must use approved matched 8-bit covers and identical payloads.

In [ ]:
SEED = 20260903
rng = np.random.default_rng(SEED)
cover = rng.integers(0, 256, size=(512, 512), dtype=np.uint8)
payload = b"BASMEDSecure disclosed baseline demonstration"

print({"shape": cover.shape, "dtype": str(cover.dtype), "payload_bytes": len(payload)})

## 3. Embedding

The result records the selected profile, predicted-label capacity, disagreement between threshold targets and predictions, fit iterations, and the external actual-width map.

In [ ]:
embedded = embed_baseline(cover, payload)
map_cost = direct_map_cost(len(embedded.depth_map))

embedding_summary = {
    "selected_profile": embedded.selected_profile,
    "raw_capacity_bits": embedded.raw_capacity_bits,
    "target_counts": embedded.target_counts,
    "predicted_counts": embedded.predicted_counts,
    "label_disagreement_count": embedded.label_disagreement_count,
    "visited_map_positions": len(embedded.depth_map),
    "direct_map_bits": map_cost.direct_symbol_bits,
}
embedding_summary

## 4. Extraction

Extraction reads exactly the widths recorded in the external map. Exact byte equality and bit error rate must be observed after execution.

In [ ]:
recovered = extract_baseline(embedded.stego, embedded.depth_map)
recovery = recovery_metrics(payload, recovered)
distortion = distortion_metrics(cover, embedded.stego, Domain(8, 0))

{
    "recovery": asdict(recovery),
    "distortion": asdict(distortion),
    "payload_preview": recovered[:64],
}

## 5. Technical visualization

The plots show the cover, stego array, and absolute stored-value difference. They describe array distortion only and do not establish diagnostic safety.

In [ ]:
difference = np.abs(cover.astype(np.int16) - embedded.stego.astype(np.int16))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cover, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Cover")
axes[1].imshow(embedded.stego, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Stego")
image = axes[2].imshow(difference, cmap="magma", vmin=0, vmax=7)
axes[2].set_title("Absolute difference")
for axis in axes:
    axis.axis("off")
fig.colorbar(image, ax=axes[2], fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## 6. Interpretation boundary

A successful run would demonstrate behavior of this disclosed baseline on the selected fixture. It would not reproduce the paper dataset, validate the original prototype, prove DICOM compatibility, establish security, or establish clinical preservation.